# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a Croissant-structured dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset Croissant schema is available at:
```
https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json
```

This dataset contains survey data on mental health indicators among residents of Kilifi County, including demographic information and common psychological assessment scores.

In [ ]:
# Install mlcroissant if not already installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and preview its content using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL (Croissant schema)
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# .metadata returns a subclass instance; access its attributes directly
print(f"Dataset name: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Let's review the available record sets, their IDs, and inspect available fields and columns for further exploration.

<sub>All entities are referenced by their `@id` as required by the Croissant specification.</sub>

In [ ]:
# List all available record sets

print('Available Record Sets:')
record_sets = dataset.metadata.record_sets

for rs in record_sets:
    print(f"- @id: {rs.id} | Name: {rs.name}")

# Let's inspect the fields/columns for each record set
print("\nSample fields per record set:")
for rs in record_sets:
    print(f"\nRecordSet: {rs.id} — {rs.name}")
    if rs.fields:
        for field in rs.fields[:5]:  # Show up to 5 fields per record set
            field_info = f"   @id: {field.id} | Name: {getattr(field, 'name', None)} | Data type: {getattr(field, 'data_type', None)}"
            print(field_info)
    else:
        print('   No fields found.')

## 3. Data Extraction
We'll extract data from each available record set using their `@id`. Each record set is loaded into a pandas DataFrame for analysis.

Below, we extract all records sets and display their first few rows (head).

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load each record set's data
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"  Columns: {list(df.columns)}\n  Number of records: {len(df)}\n")
    print(df.head(2))
    print("-"*60)
# Choose the primary record set for in-depth analysis.
# Let's pick the largest, or the one with demographic and survey data if available.
primary_record_set_id = record_set_ids[0] if record_set_ids else None

if primary_record_set_id:
    print(f"\nPrimary record set for EDA: {primary_record_set_id}")
    print(f"DataFrame shape: {dataframes[primary_record_set_id].shape}")
    print(f"Columns: {list(dataframes[primary_record_set_id].columns)}")

## 4. Exploratory Data Analysis (EDA)
Let's perform some initial EDA steps on the primary record set loaded above. Typical steps might include filtering, normalization, and grouping by demographic fields. 

*We'll illustrate by selecting numerical fields (such as `gad7_score`, `phq9_score`, or `psq_score` if available).*

In [ ]:
import numpy as np

# Choose a numeric field for analysis, referencing it by its @id (column name as loaded)
# We search for a likely field such as 'gad7_score', 'phq9_score', etc.
numeric_candidates = [c for c in dataframes[primary_record_set_id].columns if any(sc in c.lower() for sc in ['gad7', 'phq9', 'psq', 'score', 'age', 'sum'])]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # fallback to 'age' or the first numeric field
    numeric_field = 'age' if 'age' in dataframes[primary_record_set_id].columns else dataframes[primary_record_set_id].select_dtypes(include=np.number).columns[0]

print(f"Selected numeric field for analysis: {numeric_field}\n")

# Example threshold for filtering
threshold = 10

# Remove NA before filtering
df = dataframes[primary_record_set_id].copy()
df = df[df[numeric_field].notna()]
filtered_df = df[df[numeric_field] > threshold]

print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
col_normalized = f"{numeric_field}_normalized"
filtered_df[col_normalized] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, col_normalized]].head())

# Group by a likely demographic field such as 'gender', 'village', 'level_of_education', etc.
demo_keys = ['gender', 'village', 'level_of_education', 'marital_status']
possible_groups = [col for col in df.columns if any(key in col.lower() for key in demo_keys)]
if possible_groups:
    group_field = possible_groups[0]
    print(f"\nGrouping by field: {group_field}\n")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename(columns={numeric_field: f"mean_{numeric_field}"})
    print(grouped_df.head())
else:
    print("No appropriate demographic group field found for grouping.")

## 5. Visualization
Let's visualize the distribution of our chosen numeric field and, if available, compare means by a demographic field (e.g., by gender or education level).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# Boxplot by group field if available
if possible_groups:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We successfully loaded the Croissant-described mental health survey dataset using `mlcroissant`.
- Explored available record sets and identified key fields using their `@id`.
- Performed essential EDA steps such as filtering and normalization on numeric assessment scores.
- Visualized key data distributions and observed group differences (e.g., by demographic group).

**Next steps:**
- Deeper statistical analysis (e.g., by assessment instrument, correlation between scores)
- Explore relationships between demographic and mental health variables
- Prepare features for machine learning models or public health reporting

**Note:**
- Always ensure proper handling of sensitive information and compliance with dataset's license and ethical guidelines.